# Segmentation Comparison: BLAST vs UMAP vs PCA on Histopathology Images

This notebook compares three **patch-based segmentation methods** on the **OCDC oral cancer histopathology dataset**:

1. **Multi-Scale BLAST** (8×8 + 16×16 + 32×32) — cosine similarity + weighted voting across scales
2. **UMAP Segmentation** (16×16) — UMAP dimensionality reduction + KNN voting in reduced space
3. **PCA Segmentation** (16×16) — PCA dimensionality reduction + KNN voting in reduced space

All three methods produce **pixel-level tumor segmentation masks** and are evaluated with Dice Score, IoU, Accuracy, Precision, Recall, and F1.

> **Why not t-SNE?** sklearn's t-SNE has no `transform()` method — it cannot project unseen patches, making it unusable for segmentation inference. See the explanation at the end of this notebook.

## Cell 1 — Install & Import Libraries

In [ ]:
import subprocess, sys
for pkg in ['umap-learn', 'gdown']:
    try:
        __import__(pkg.replace('-', '_').split('_')[0] if pkg != 'umap-learn' else 'umap')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os, random, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    jaccard_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize as l2_normalize
import umap

warnings.filterwarnings('ignore')
import matplotlib.patches as mpatches
import pandas as pd
sns.set_style('whitegrid')
print('All libraries imported successfully.')

## Cell 2 — Load OCDC Dataset (Same as BLAST Notebook)

In [ ]:
SEED = 42
np.random.seed(SEED); random.seed(SEED)
IMG_SIZE = 256

EXTRACT_DIR = None
GDRIVE_FILE_ID = '1d7FPc3jAUeNwIfOOG2sC-RlvCe9Pfb_P'

# Search for dataset locally
search_paths = [
    '/kaggle/input', '/content/dataset',
    '/mnt/c/Users/Himanshu Parashar/Desktop/histo',
    os.path.expanduser('~'),
]
for sp in search_paths:
    if not os.path.isdir(sp): continue
    for root, dirs, files in os.walk(sp):
        if '01-original' in dirs and '02-mask' in dirs:
            parent = root
            if os.path.isdir(os.path.join(parent, '01-original')):
                EXTRACT_DIR = parent
                break
    if EXTRACT_DIR: break

if EXTRACT_DIR:
    print(f'Dataset found locally: {EXTRACT_DIR}')
else:
    import gdown, zipfile
    zip_path = '/tmp/ocdc_dataset.zip'
    extract_to = '/tmp/ocdc_dataset'
    if not os.path.exists(zip_path):
        gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}', zip_path, quiet=False)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    for root, dirs, files in os.walk(extract_to):
        if '01-original' in dirs and '02-mask' in dirs:
            EXTRACT_DIR = root; break
    print(f'Dataset downloaded: {EXTRACT_DIR}')

## Cell 3 — Pair Images with Masks & Load

In [ ]:
# Pair images with masks
paired = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    if os.path.basename(root) == '01-original':
        mask_dir = os.path.join(os.path.dirname(root), '02-mask')
        if os.path.isdir(mask_dir):
            for f in sorted(files):
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
                    img_path = os.path.join(root, f)
                    mask_path = os.path.join(mask_dir, f)
                    if os.path.exists(mask_path):
                        paired.append((img_path, mask_path))
print(f'Total paired samples: {len(paired)}')

# Load images and masks
random.shuffle(paired)
images, masks = [], []
for img_path, mask_path in paired[:100]:
    img = cv2.imread(img_path)
    if img is None: continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.float32)
    images.append(img.astype(np.float32) / 255.0)
    masks.append(mask)

images = np.array(images)
masks = np.array(masks)
NUM_IMAGES = len(images)
print(f'Loaded {NUM_IMAGES} images ({IMG_SIZE}x{IMG_SIZE}) | Tumor: {masks.mean()*100:.1f}%')

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(images, masks, test_size=0.2, random_state=SEED)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## Segmentation Methods

We define three patch-based segmentation engines, each with the same interface:
- `build_database(train_images, train_masks)` — build a reference database from training data
- `generate_maps(image, stride)` — produce a probability map for one test image

**Method 1: Multi-Scale BLAST** — cosine similarity at 3 scales (8×8, 16×16, 32×32) with learned weights
**Method 2: UMAP Segmentation** — UMAP reduction to 20D + KNN voting in reduced space
**Method 3: PCA Segmentation** — PCA reduction to 50D + KNN voting in reduced space

In [ ]:
# ── Method 1: Multi-Scale BLAST Segmentation ──
NUM_CLASSES = 2

class BLASTSegEngine:
    """Single-scale BLAST for segmentation (generates probability maps)."""
    def __init__(self, window_size=16, top_k=5, db_stride=8):
        self.ws = window_size
        self.top_k = top_k
        self.db_stride = db_stride
        self.db_features = None
        self.db_labels = None

    def build_database(self, train_images, train_masks):
        ws = self.ws
        all_feats, all_labels = [], []
        for img, mask in zip(train_images, train_masks):
            h, w = img.shape[:2]
            if img.max() > 1.0:
                img = img.astype(np.float32) / 255.0
            for row in range(0, h - ws + 1, self.db_stride):
                for col in range(0, w - ws + 1, self.db_stride):
                    patch = img[row:row+ws, col:col+ws, :].flatten()
                    cr, cc = row + ws // 2, col + ws // 2
                    all_feats.append(patch)
                    all_labels.append(int(mask[cr, cc] > 0.5))
        self.db_features = l2_normalize(np.array(all_feats, dtype=np.float32))
        self.db_labels = np.array(all_labels)
        print(f'    Scale {ws}x{ws}: {len(self.db_labels)} DB patches')

    def generate_maps(self, image, stride=4):
        h, w = image.shape[:2]
        ws = self.ws
        if image.max() > 1.0:
            image = image.astype(np.float32) / 255.0
        grid_ys = np.arange(0, h - ws + 1, stride)
        grid_xs = np.arange(0, w - ws + 1, stride)
        gh, gw = len(grid_ys), len(grid_xs)
        score_maps = np.zeros((gh, gw, NUM_CLASSES), dtype=np.float32)
        for ri, y in enumerate(grid_ys):
            row_feats = []
            for x in grid_xs:
                patch = image[y:y+ws, x:x+ws, :].flatten()
                row_feats.append(patch)
            row_feats = l2_normalize(np.array(row_feats, dtype=np.float32))
            sims = row_feats @ self.db_features.T
            top_idx = np.argpartition(sims, -self.top_k, axis=1)[:, -self.top_k:]
            top_sims = np.maximum(np.take_along_axis(sims, top_idx, axis=1), 0)
            top_labels = self.db_labels[top_idx]
            for c in range(NUM_CLASSES):
                score_maps[ri, :, c] = (top_sims * (top_labels == c)).sum(axis=1)
        total = score_maps.sum(axis=-1, keepdims=True) + 1e-10
        score_maps = score_maps / total
        full_maps = np.zeros((h, w, NUM_CLASSES), dtype=np.float32)
        for c in range(NUM_CLASSES):
            full_maps[:, :, c] = cv2.resize(score_maps[:, :, c], (w, h),
                                            interpolation=cv2.INTER_LINEAR)
        return full_maps


class MultiScaleBLAST:
    def __init__(self, scales=[8, 16, 32], top_k=5, db_stride=8):
        self.scales = scales
        self.engines = {s: BLASTSegEngine(window_size=s, top_k=top_k, db_stride=db_stride)
                        for s in scales}
        self.weights = None

    def build_databases(self, train_images, train_masks):
        print('Building multi-scale BLAST databases...')
        for s, eng in self.engines.items():
            eng.build_database(train_images, train_masks)

    def grid_search_weights(self, val_images, val_masks, stride=8):
        print('Grid searching optimal scale weights...')
        scale_maps = {}
        for s, eng in self.engines.items():
            scale_maps[s] = [eng.generate_maps(img, stride=stride) for img in val_images]
        best_dice, best_w = 0, (0.33, 0.34, 0.33)
        for w1 in np.arange(0.1, 0.9, 0.1):
            for w2 in np.arange(0.1, 1.0 - w1, 0.1):
                w3 = round(1.0 - w1 - w2, 2)
                if w3 < 0.05: continue
                preds = []
                for i in range(len(val_images)):
                    combined = (w1 * scale_maps[self.scales[0]][i]
                              + w2 * scale_maps[self.scales[1]][i]
                              + w3 * scale_maps[self.scales[2]][i])
                    preds.append(np.argmax(combined, axis=-1).astype(np.uint8))
                all_true = np.concatenate([m.flatten() for m in val_masks])
                all_pred = np.concatenate([p.flatten() for p in preds])
                dice_scores = []
                for c in range(NUM_CLASSES):
                    gt_c = (all_true == c); pr_c = (all_pred == c)
                    inter = (gt_c & pr_c).sum()
                    dice_scores.append(2 * inter / (gt_c.sum() + pr_c.sum() + 1e-10))
                dice = np.mean(dice_scores)
                if dice > best_dice:
                    best_dice = dice; best_w = (w1, w2, w3)
        self.weights = best_w
        print(f'  Best weights: w8={best_w[0]:.2f}, w16={best_w[1]:.2f}, w32={best_w[2]:.2f} (Dice={best_dice:.4f})')

    def segment(self, image, stride=4):
        maps = [self.engines[s].generate_maps(image, stride=stride) for s in self.scales]
        combined = self.weights[0]*maps[0] + self.weights[1]*maps[1] + self.weights[2]*maps[2]
        return np.argmax(combined, axis=-1).astype(np.uint8), combined

print('BLAST segmentation classes defined.')

In [ ]:
# ── Method 2: UMAP Segmentation Engine ──

class UMAPSegEngine:
    """Single-scale UMAP-based segmentation.

    Fits UMAP on flattened DB patches to reduce dimensionality,
    then uses KNN weighted voting in the reduced space.
    """
    def __init__(self, window_size=16, n_components=20, n_neighbors=15, top_k=5, db_stride=8):
        self.ws = window_size
        self.n_components = n_components
        self.n_neighbors = n_neighbors
        self.top_k = top_k
        self.db_stride = db_stride
        self.reducer = None
        self.db_reduced = None
        self.db_labels = None

    def build_database(self, train_images, train_masks):
        ws = self.ws
        all_feats, all_labels = [], []
        for img, mask in zip(train_images, train_masks):
            h, w = img.shape[:2]
            if img.max() > 1.0:
                img = img.astype(np.float32) / 255.0
            for row in range(0, h - ws + 1, self.db_stride):
                for col in range(0, w - ws + 1, self.db_stride):
                    patch = img[row:row+ws, col:col+ws, :].flatten()
                    cr, cc = row + ws // 2, col + ws // 2
                    all_feats.append(patch)
                    all_labels.append(int(mask[cr, cc] > 0.5))
        raw_feats = np.array(all_feats, dtype=np.float32)
        self.db_labels = np.array(all_labels)

        # Fit UMAP on DB patches
        print(f'    UMAP: fitting on {len(raw_feats)} patches ({raw_feats.shape[1]}D -> {self.n_components}D)...')
        self.reducer = umap.UMAP(
            n_components=self.n_components,
            n_neighbors=self.n_neighbors,
            min_dist=0.0,
            metric='euclidean',
            random_state=42,
        )
        self.db_reduced = self.reducer.fit_transform(raw_feats)
        # L2-normalize for cosine similarity
        self.db_reduced = l2_normalize(self.db_reduced)
        print(f'    UMAP DB: {len(self.db_labels)} patches in {self.n_components}D')

    def generate_maps(self, image, stride=4):
        h, w = image.shape[:2]
        ws = self.ws
        if image.max() > 1.0:
            image = image.astype(np.float32) / 255.0
        grid_ys = np.arange(0, h - ws + 1, stride)
        grid_xs = np.arange(0, w - ws + 1, stride)
        gh, gw = len(grid_ys), len(grid_xs)
        score_maps = np.zeros((gh, gw, NUM_CLASSES), dtype=np.float32)

        # Extract all query patches
        query_feats = []
        for y in grid_ys:
            for x in grid_xs:
                patch = image[y:y+ws, x:x+ws, :].flatten()
                query_feats.append(patch)
        query_feats = np.array(query_feats, dtype=np.float32)

        # Transform through UMAP
        query_reduced = self.reducer.transform(query_feats)
        query_reduced = l2_normalize(query_reduced)

        # Cosine similarity against DB
        sims = query_reduced @ self.db_reduced.T
        top_idx = np.argpartition(sims, -self.top_k, axis=1)[:, -self.top_k:]
        top_sims = np.maximum(np.take_along_axis(sims, top_idx, axis=1), 0)
        top_labels = self.db_labels[top_idx]

        for c in range(NUM_CLASSES):
            class_scores = (top_sims * (top_labels == c)).sum(axis=1)
            score_maps[:, :, c] = class_scores.reshape(gh, gw)

        total = score_maps.sum(axis=-1, keepdims=True) + 1e-10
        score_maps = score_maps / total
        full_maps = np.zeros((h, w, NUM_CLASSES), dtype=np.float32)
        for c in range(NUM_CLASSES):
            full_maps[:, :, c] = cv2.resize(score_maps[:, :, c], (w, h),
                                            interpolation=cv2.INTER_LINEAR)
        return full_maps

print('UMAP segmentation engine defined.')

In [ ]:
# ── Method 3: PCA Segmentation Engine ──

class PCASegEngine:
    """Single-scale PCA-based segmentation.

    Fits PCA on flattened DB patches to reduce dimensionality,
    then uses KNN weighted voting in the reduced space.
    """
    def __init__(self, window_size=16, n_components=50, top_k=5, db_stride=8):
        self.ws = window_size
        self.n_components = n_components
        self.top_k = top_k
        self.db_stride = db_stride
        self.pca = None
        self.db_reduced = None
        self.db_labels = None

    def build_database(self, train_images, train_masks):
        ws = self.ws
        all_feats, all_labels = [], []
        for img, mask in zip(train_images, train_masks):
            h, w = img.shape[:2]
            if img.max() > 1.0:
                img = img.astype(np.float32) / 255.0
            for row in range(0, h - ws + 1, self.db_stride):
                for col in range(0, w - ws + 1, self.db_stride):
                    patch = img[row:row+ws, col:col+ws, :].flatten()
                    cr, cc = row + ws // 2, col + ws // 2
                    all_feats.append(patch)
                    all_labels.append(int(mask[cr, cc] > 0.5))
        raw_feats = np.array(all_feats, dtype=np.float32)
        self.db_labels = np.array(all_labels)

        # Fit PCA on DB patches
        print(f'    PCA: fitting on {len(raw_feats)} patches ({raw_feats.shape[1]}D -> {self.n_components}D)...')
        self.pca = PCA(n_components=self.n_components, random_state=42)
        self.db_reduced = self.pca.fit_transform(raw_feats)
        # L2-normalize for cosine similarity
        self.db_reduced = l2_normalize(self.db_reduced)
        var_explained = self.pca.explained_variance_ratio_.sum() * 100
        print(f'    PCA DB: {len(self.db_labels)} patches in {self.n_components}D ({var_explained:.1f}% variance)')

    def generate_maps(self, image, stride=4):
        h, w = image.shape[:2]
        ws = self.ws
        if image.max() > 1.0:
            image = image.astype(np.float32) / 255.0
        grid_ys = np.arange(0, h - ws + 1, stride)
        grid_xs = np.arange(0, w - ws + 1, stride)
        gh, gw = len(grid_ys), len(grid_xs)
        score_maps = np.zeros((gh, gw, NUM_CLASSES), dtype=np.float32)

        # Extract all query patches
        query_feats = []
        for y in grid_ys:
            for x in grid_xs:
                patch = image[y:y+ws, x:x+ws, :].flatten()
                query_feats.append(patch)
        query_feats = np.array(query_feats, dtype=np.float32)

        # Transform through PCA
        query_reduced = self.pca.transform(query_feats)
        query_reduced = l2_normalize(query_reduced)

        # Cosine similarity against DB
        sims = query_reduced @ self.db_reduced.T
        top_idx = np.argpartition(sims, -self.top_k, axis=1)[:, -self.top_k:]
        top_sims = np.maximum(np.take_along_axis(sims, top_idx, axis=1), 0)
        top_labels = self.db_labels[top_idx]

        for c in range(NUM_CLASSES):
            class_scores = (top_sims * (top_labels == c)).sum(axis=1)
            score_maps[:, :, c] = class_scores.reshape(gh, gw)

        total = score_maps.sum(axis=-1, keepdims=True) + 1e-10
        score_maps = score_maps / total
        full_maps = np.zeros((h, w, NUM_CLASSES), dtype=np.float32)
        for c in range(NUM_CLASSES):
            full_maps[:, :, c] = cv2.resize(score_maps[:, :, c], (w, h),
                                            interpolation=cv2.INTER_LINEAR)
        return full_maps

print('PCA segmentation engine defined.')

## Build Databases & Find Optimal Weights

- **BLAST**: Build 3-scale databases (8×8, 16×16, 32×32) + grid search scale weights on validation set
- **UMAP**: Build single-scale (16×16) database with UMAP reduction to 20D
- **PCA**: Build single-scale (16×16) database with PCA reduction to 50D

In [ ]:
# ── Use ALL training images for DB; small subset for BLAST weight validation ──
# DB gets every training image (maximizes patch database size)
db_images = list(X_train)
db_masks = list(y_train)

# Validation for BLAST weight search: use a few training images (overlap with DB is OK
# for weight tuning — we're only choosing between scale weight combos, not fitting a model)
n_val = min(5, len(X_train))
val_idx = np.random.choice(len(X_train), n_val, replace=False)
val_images = [X_train[i] for i in val_idx]
val_masks = [y_train[i] for i in val_idx]
print(f'DB images: {len(db_images)} | Validation images: {n_val} | Test images: {len(X_test)}')

# ── Build BLAST databases & find optimal weights ──
print('\n' + '='*60)
print('  Building BLAST databases')
print('='*60)
blast_seg = MultiScaleBLAST(scales=[8, 16, 32], top_k=5, db_stride=8)
blast_seg.build_databases(db_images, db_masks)
blast_seg.grid_search_weights(val_images, val_masks, stride=8)

# ── Build UMAP database ──
print('\n' + '='*60)
print('  Building UMAP database')
print('='*60)
umap_seg = UMAPSegEngine(window_size=16, n_components=20, n_neighbors=15, top_k=5, db_stride=8)
umap_seg.build_database(db_images, db_masks)

# ── Build PCA database ──
print('\n' + '='*60)
print('  Building PCA database')
print('='*60)
pca_seg = PCASegEngine(window_size=16, n_components=50, top_k=5, db_stride=8)
pca_seg.build_database(db_images, db_masks)

print('\nAll 3 segmentation engines ready.')

## Run All Methods on Test Images

Segment every test image with all three methods and compute per-method metrics:
**Dice Score**, **IoU**, **Pixel Accuracy**, **Precision**, **Recall**, **F1**

In [ ]:
def compute_seg_metrics(y_true_list, y_pred_list):
    """Compute segmentation metrics from lists of masks."""
    all_true = np.concatenate([m.flatten() for m in y_true_list]).astype(np.uint8)
    all_pred = np.concatenate([m.flatten() for m in y_pred_list]).astype(np.uint8)

    pixel_acc = accuracy_score(all_true, all_pred)
    mean_iou = jaccard_score(all_true, all_pred, average='macro', zero_division=0)
    precision = precision_score(all_true, all_pred, average='macro', zero_division=0)
    recall_val = recall_score(all_true, all_pred, average='macro', zero_division=0)
    f1_val = f1_score(all_true, all_pred, average='macro', zero_division=0)

    # Per-class Dice
    dice_scores = []
    for c in range(NUM_CLASSES):
        gt_c = (all_true == c); pr_c = (all_pred == c)
        inter = (gt_c & pr_c).sum()
        dice_scores.append(2 * inter / (gt_c.sum() + pr_c.sum() + 1e-10))
    mean_dice = np.mean(dice_scores)

    return {
        'Dice': mean_dice,
        'IoU': mean_iou,
        'Accuracy': pixel_acc,
        'Precision': precision,
        'Recall': recall_val,
        'F1': f1_val,
        'Tumor Dice': dice_scores[1],
        'Normal Dice': dice_scores[0],
    }, all_true, all_pred

# ── Run BLAST on all test images ──
print('Running Multi-Scale BLAST segmentation...')
blast_preds = []
t0 = time.time()
for i in range(len(X_test)):
    mask_pred, _ = blast_seg.segment(X_test[i], stride=4)
    blast_preds.append(mask_pred)
    if (i+1) % 5 == 0: print(f'  BLAST: {i+1}/{len(X_test)}')
blast_time = time.time() - t0
print(f'  BLAST done in {blast_time:.1f}s')

# ── Run UMAP on all test images ──
print('\nRunning UMAP segmentation...')
umap_preds = []
t0 = time.time()
for i in range(len(X_test)):
    prob_map = umap_seg.generate_maps(X_test[i], stride=4)
    umap_preds.append(np.argmax(prob_map, axis=-1).astype(np.uint8))
    if (i+1) % 5 == 0: print(f'  UMAP: {i+1}/{len(X_test)}')
umap_time = time.time() - t0
print(f'  UMAP done in {umap_time:.1f}s')

# ── Run PCA on all test images ──
print('\nRunning PCA segmentation...')
pca_preds = []
t0 = time.time()
for i in range(len(X_test)):
    prob_map = pca_seg.generate_maps(X_test[i], stride=4)
    pca_preds.append(np.argmax(prob_map, axis=-1).astype(np.uint8))
    if (i+1) % 5 == 0: print(f'  PCA: {i+1}/{len(X_test)}')
pca_time = time.time() - t0
print(f'  PCA done in {pca_time:.1f}s')

# ── Compute metrics ──
y_test_int = [m.astype(np.uint8) for m in y_test]
blast_metrics, blast_true, blast_pred_flat = compute_seg_metrics(y_test_int, blast_preds)
umap_metrics, umap_true, umap_pred_flat = compute_seg_metrics(y_test_int, umap_preds)
pca_metrics, pca_true, pca_pred_flat = compute_seg_metrics(y_test_int, pca_preds)

print('\nAll methods complete. Metrics computed.')

In [ ]:
# ── Results Comparison Table ──
methods_data = {
    'BLAST (Multi-Scale)': blast_metrics,
    'UMAP (16×16)': umap_metrics,
    'PCA (16×16)': pca_metrics,
}

rows = []
for name, metrics in methods_data.items():
    rows.append({
        'Method': name,
        'Dice': f'{metrics["Dice"]:.4f}',
        'IoU': f'{metrics["IoU"]:.4f}',
        'Accuracy': f'{metrics["Accuracy"]:.4f}',
        'Precision': f'{metrics["Precision"]:.4f}',
        'Recall': f'{metrics["Recall"]:.4f}',
        'F1': f'{metrics["F1"]:.4f}',
        'Tumor Dice': f'{metrics["Tumor Dice"]:.4f}',
    })

df_results = pd.DataFrame(rows)
print('='*90)
print('  SEGMENTATION COMPARISON: BLAST vs UMAP vs PCA')
print('='*90)
print(df_results.to_string(index=False))
print('='*90)

# Highlight best per metric
print('\nBest method per metric:')
for metric in ['Dice', 'IoU', 'Accuracy', 'Precision', 'Recall', 'F1', 'Tumor Dice']:
    vals = {name: m[metric] for name, m in methods_data.items()}
    best = max(vals, key=vals.get)
    print(f'  {metric:>12s}: {best} ({vals[best]:.4f})')

In [ ]:
# ── Bar Chart: All 3 Methods Across Metrics ──
metric_names = ['Dice', 'IoU', 'Accuracy', 'Precision', 'Recall', 'F1']
method_names = ['BLAST\n(Multi-Scale)', 'UMAP\n(16×16)', 'PCA\n(16×16)']
all_metrics = [blast_metrics, umap_metrics, pca_metrics]
colors_bar = ['#E53935', '#1E88E5', '#43A047']

fig, axes = plt.subplots(1, len(metric_names), figsize=(20, 5))

for j, metric in enumerate(metric_names):
    vals = [m[metric] for m in all_metrics]
    bars = axes[j].bar(method_names, vals, color=colors_bar, edgecolor='white', linewidth=1.5)
    axes[j].set_title(metric, fontsize=13, fontweight='bold')
    axes[j].set_ylim(0, 1.05)
    for bar, v in zip(bars, vals):
        axes[j].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Segmentation Quality: BLAST vs UMAP vs PCA',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Side-by-Side Visualization: Original | GT | BLAST | UMAP | PCA ──
# Pick test images with most tumor content for visual impact
tumor_content = [y_test[i].mean() for i in range(len(y_test))]
n_show = min(5, len(X_test))
show_idx = np.argsort(tumor_content)[::-1][:n_show]

def mask_to_color(mask):
    """Convert binary mask to RGB: red=tumor, green=normal."""
    color = np.zeros((*mask.shape[:2], 3), dtype=np.uint8)
    m = mask.astype(np.uint8) if mask.ndim == 2 else mask[:, :, 0].astype(np.uint8)
    color[m == 1] = [255, 99, 71]    # red = tumor
    color[m == 0] = [144, 238, 144]  # green = normal
    return color

fig, axes = plt.subplots(n_show, 5, figsize=(22, 4.5 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]  # ensure 2D indexing
col_titles = ['H&E Image', 'Ground Truth', 'BLAST', 'UMAP', 'PCA']

for row, idx in enumerate(show_idx):
    # Original image
    axes[row, 0].imshow(X_test[idx])
    axes[row, 0].axis('off')

    # Ground truth
    axes[row, 1].imshow(mask_to_color(y_test[idx]))
    axes[row, 1].axis('off')

    # BLAST prediction
    axes[row, 2].imshow(mask_to_color(blast_preds[idx]))
    axes[row, 2].axis('off')

    # UMAP prediction
    axes[row, 3].imshow(mask_to_color(umap_preds[idx]))
    axes[row, 3].axis('off')

    # PCA prediction
    axes[row, 4].imshow(mask_to_color(pca_preds[idx]))
    axes[row, 4].axis('off')

    # Per-image Dice for each method
    gt_flat = y_test[idx].flatten().astype(np.uint8)
    for col_i, pred in enumerate([blast_preds[idx], umap_preds[idx], pca_preds[idx]]):
        pred_flat = pred.flatten().astype(np.uint8)
        inter = ((gt_flat == 1) & (pred_flat == 1)).sum()
        dice_i = 2 * inter / ((gt_flat == 1).sum() + (pred_flat == 1).sum() + 1e-10)
        axes[row, col_i + 2].set_xlabel(f'Dice={dice_i:.3f}', fontsize=10, fontweight='bold')

# Column titles
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=13, fontweight='bold')

legend_patches = [
    mpatches.Patch(color=np.array([144,238,144])/255, label='Normal'),
    mpatches.Patch(color=np.array([255,99,71])/255, label='Tumor'),
]
fig.legend(handles=legend_patches, loc='upper center', ncol=2, fontsize=12,
           bbox_to_anchor=(0.5, 1.01))
fig.suptitle(f'Segmentation Comparison: BLAST vs UMAP vs PCA (Top-{n_show} Tumor Images)',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrices for All 3 Methods ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

method_cm_data = [
    ('BLAST (Multi-Scale)', blast_true, blast_pred_flat),
    ('UMAP (16×16)', umap_true, umap_pred_flat),
    ('PCA (16×16)', pca_true, pca_pred_flat),
]

for i, (name, y_true, y_pred) in enumerate(method_cm_data):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-10)

    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', ax=axes[i],
                xticklabels=['Normal', 'Tumor'], yticklabels=['Normal', 'Tumor'],
                vmin=0, vmax=1)
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('True Label')
    axes[i].set_xlabel('Predicted Label')

    # Add raw counts as text below
    tn, fp, fn, tp = cm.ravel()
    axes[i].text(0.5, -0.15, f'TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}',
                 transform=axes[i].transAxes, ha='center', fontsize=9, style='italic')

plt.suptitle('Normalized Confusion Matrices — All Methods',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Why Not t-SNE for Segmentation?

We deliberately **excluded t-SNE** as a segmentation method. Here's why:

### 1. No `transform()` Method
sklearn's `TSNE` only has `fit_transform()` — it **cannot project new/unseen data** into the learned embedding.
To segment a test image, we need to transform its patches through the same reduction learned on the database.
UMAP has `transform()`, PCA has `transform()`, but t-SNE does not.

### 2. Computational Complexity
- t-SNE is **O(n²)** in memory and time (pairwise affinities)
- For segmentation, each test image produces ~3,600 query patches (256×256 image, 16×16 patches, stride 4)
- With a database of ~50,000+ patches, t-SNE would need to refit on the combined set every time — infeasible

### 3. Designed for Visualization, Not Inference
t-SNE optimizes for **visual cluster separation in 2D/3D**. It:
- Does not preserve global distances (only local neighborhoods)
- Produces different embeddings on different runs (non-deterministic structure)
- Is meant for **human interpretation**, not downstream classification

### Summary
| Property | t-SNE | UMAP | PCA |
|----------|-------|------|-----|
| `transform()` for new data | No | Yes | Yes |
| Complexity | O(n²) | O(n·k) | O(n·d) |
| Preserves global structure | No | Partially | Yes |
| Suitable for segmentation | **No** | **Yes** | **Yes** |

## Summary & Key Findings

### Methods Compared
| Method | Scale | Dimensionality | Approach |
|--------|-------|---------------|----------|
| **Multi-Scale BLAST** | 8×8 + 16×16 + 32×32 | Raw 768D (per scale) | Cosine similarity + weighted voting across 3 scales |
| **UMAP Segmentation** | 16×16 | 768D → 20D | UMAP reduction + cosine KNN voting in 20D |
| **PCA Segmentation** | 16×16 | 768D → 50D | PCA reduction + cosine KNN voting in 50D |

### Key Observations
1. **BLAST benefits from multi-scale context** — combining fine (8×8) and coarse (32×32) patterns captures both cellular and tissue-level features
2. **UMAP preserves nonlinear structure** — its manifold-aware reduction can separate classes that are nonlinearly entangled in pixel space
3. **PCA is the fastest** but limited to linear projections — it serves as a strong baseline
4. **t-SNE is not viable** for segmentation due to lack of `transform()` and O(n²) cost
5. All three methods work **without any neural network training** — they are purely pattern-matching approaches suitable for small datasets

In [ ]:
# ── Final Results Summary ──
print('='*70)
print('  FINAL RESULTS — Segmentation Comparison')
print('='*70)
print(f'  Dataset: OCDC oral cancer histopathology ({len(images)} images)')
print(f'  Train: {len(X_train)} | Test: {len(X_test)}')
print(f'  Image size: {IMG_SIZE}x{IMG_SIZE}')
print()

methods_summary = [
    ('BLAST (Multi-Scale 8+16+32)', blast_metrics, blast_time,
     f'weights: w8={blast_seg.weights[0]:.2f}, w16={blast_seg.weights[1]:.2f}, w32={blast_seg.weights[2]:.2f}'),
    ('UMAP (16×16, 20D)', umap_metrics, umap_time,
     f'n_neighbors=15, top_k=5'),
    ('PCA (16×16, 50D)', pca_metrics, pca_time,
     f'top_k=5'),
]

for name, metrics, t_sec, config in methods_summary:
    print(f'  {name}')
    print(f'    Config:    {config}')
    print(f'    Dice:      {metrics["Dice"]:.4f}  |  IoU: {metrics["IoU"]:.4f}  |  Accuracy: {metrics["Accuracy"]:.4f}')
    print(f'    Precision: {metrics["Precision"]:.4f}  |  Recall: {metrics["Recall"]:.4f}  |  F1: {metrics["F1"]:.4f}')
    print(f'    Tumor Dice: {metrics["Tumor Dice"]:.4f}  |  Time: {t_sec:.1f}s')
    print()

# Determine best method
best_method = max(methods_summary, key=lambda x: x[1]['Dice'])
print(f'  Best overall (by Dice): {best_method[0]} — Dice={best_method[1]["Dice"]:.4f}')
print('='*70)
print()
print('t-SNE was excluded because sklearn TSNE has no transform() method,')
print('making it unable to project unseen test patches for segmentation.')